##Widgets + Setup:

In [0]:
import logging
import pyspark.sql.functions as F
from pyspark.sql.window import Window

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

dbutils.widgets.text("gold_path", "abfss://gold@stnyctaxisri01.dfs.core.windows.net/nyc_taxi/")
dbutils.widgets.text("last_watermark", "1900-01-01T00:00:00Z")

gold_path      = dbutils.widgets.get("gold_path")
last_watermark = dbutils.widgets.get("last_watermark")

SILVER_TABLE = "nyc_taxi_prod.silver.trips"

logger.info(f"Gold path      : {gold_path}")
logger.info(f"Last watermark : {last_watermark if last_watermark else 'FULL LOAD'}")
logger.info(f"Silver source  : {SILVER_TABLE}")

2026-04-18 18:21:44,318 [INFO] Gold path      : abfss://gold@stnyctaxisri01.dfs.core.windows.net/nyc_taxi/
2026-04-18 18:21:44,323 [INFO] Last watermark : 1900-01-01T00:00:00Z
2026-04-18 18:21:44,325 [INFO] Silver source  : nyc_taxi_prod.silver.trips


## Read Silver with Incremental Filter:


In [0]:
logger.info(f"Reading silver table: {SILVER_TABLE}")

try:
    df_silver = spark.table(SILVER_TABLE)
    
    # Incremental filter
    df_silver = df_silver.filter(F.col("updated_at") > last_watermark)
    
    row_count = df_silver.count()
    logger.info(f"Silver rows after incremental filter: {row_count}")

except Exception as e:
    logger.error(f"Failed to read silver table: {str(e)}")
    raise

2026-04-18 18:23:05,867 [INFO] Reading silver table: nyc_taxi_prod.silver.trips
2026-04-18 18:23:07,071 [INFO] Silver rows after incremental filter: 141


##Empty DataFrame Guard:

In [0]:
import json

if row_count == 0:
    logger.warning("No new silver data — exiting early")
    dbutils.notebook.exit("NO_DATA")

logger.info("Silver data available — proceeding to gold aggregations")

2026-04-18 18:23:48,957 [INFO] Silver data available — proceeding to gold aggregations


## All 4 Gold Aggregations:

In [0]:
from pyspark.sql.functions import (
    col, count, sum, avg, max, min,
    hour, to_date, round
)

logger.info("Building gold aggregations")

# 1 — Daily Trip Summary
df_daily = df_silver \
    .withColumn("trip_date", F.to_date("pickup_datetime")) \
    .groupBy("trip_date") \
    .agg(
        F.count("trip_id").alias("total_trips"),
        F.round(F.sum("fare_amount"), 2).alias("total_revenue"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
        F.round(F.sum("tip_amount"), 2).alias("total_tips"),
        F.max("updated_at").alias("updated_at")
    )

logger.info(f"daily_trip_summary rows: {df_daily.count()}")

# 2 — Vendor Performance
df_vendor = df_silver \
    .withColumn("trip_date", F.to_date("pickup_datetime")) \
    .groupBy("vendor_id", "trip_date") \
    .agg(
        F.count("trip_id").alias("total_trips"),
        F.round(F.sum("fare_amount"), 2).alias("total_revenue"),
        F.round(F.avg("tip_amount"), 2).alias("avg_tip"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
        F.max("updated_at").alias("updated_at")
    )

logger.info(f"vendor_performance rows: {df_vendor.count()}")

# 3 — Payment Analysis
df_payment = df_silver \
    .withColumn("trip_date", F.to_date("pickup_datetime")) \
    .groupBy("payment_type", "trip_date") \
    .agg(
        F.count("trip_id").alias("total_trips"),
        F.round(F.sum("fare_amount"), 2).alias("total_revenue"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.sum("tip_amount"), 2).alias("total_tips"),
        F.max("updated_at").alias("updated_at")
    )

logger.info(f"payment_analysis rows: {df_payment.count()}")

# 4 — Hourly Demand
df_hourly = df_silver \
    .withColumn("trip_date", F.to_date("pickup_datetime")) \
    .withColumn("hour_of_day", F.hour("pickup_datetime")) \
    .groupBy("trip_date", "hour_of_day") \
    .agg(
        F.count("trip_id").alias("total_trips"),
        F.round(F.avg("passenger_count"), 1).alias("avg_passengers"),
        F.round(F.avg("trip_distance"), 2).alias("avg_distance"),
        F.max("updated_at").alias("updated_at")
    ) \
    .orderBy("trip_date", "hour_of_day")

logger.info(f"hourly_demand rows: {df_hourly.count()}")

logger.info("All 4 gold aggregations built successfully")

2026-04-18 18:25:56,727 [INFO] Python Server ready to receive messages
2026-04-18 18:25:56,728 [INFO] Received command c on object id p0
2026-04-18 18:25:56,746 [INFO] Building gold aggregations
2026-04-18 18:25:57,469 [INFO] daily_trip_summary rows: 3
2026-04-18 18:25:57,806 [INFO] vendor_performance rows: 6
2026-04-18 18:25:58,137 [INFO] payment_analysis rows: 6
2026-04-18 18:25:58,514 [INFO] hourly_demand rows: 11
2026-04-18 18:25:58,516 [INFO] All 4 gold aggregations built successfully


## Write All 4 Gold Tables as Delta:

In [0]:
def write_gold_table(df, table_name, partition_col, gold_path):
    full_table_name = f"nyc_taxi_prod.gold.{table_name}"
    table_path = f"{gold_path}{table_name}/"
    
    logger.info(f"Writing gold table: {full_table_name}")
    
    if spark.catalog.tableExists(full_table_name):
        # Get date range from current batch
        date_range = df.select(
            F.min(partition_col).alias("min_date"),
            F.max(partition_col).alias("max_date")
        ).collect()[0]
        
        min_date = str(date_range["min_date"])
        max_date = str(date_range["max_date"])
        
        logger.info(f"Overwriting partitions from {min_date} to {max_date}")
        
        df.write \
            .format("delta") \
            .mode("overwrite") \
            .option("replaceWhere", f"{partition_col} >= '{min_date}' AND {partition_col} <= '{max_date}'") \
            .option("overwriteSchema", "false") \
            .partitionBy(partition_col) \
            .save(table_path)
            
        logger.info(f"Overwrite complete: {full_table_name}")
    else:
        df.write \
            .format("delta") \
            .mode("overwrite") \
            .partitionBy(partition_col) \
            .option("path", table_path) \
            .saveAsTable(full_table_name)
        logger.info(f"Created new table: {full_table_name}")

# Write all 4 tables
write_gold_table(df_daily,   "daily_trip_summary",  "trip_date", gold_path)
write_gold_table(df_vendor,  "vendor_performance",  "trip_date", gold_path)
write_gold_table(df_payment, "payment_analysis",    "trip_date", gold_path)
write_gold_table(df_hourly,  "hourly_demand",       "trip_date", gold_path)

logger.info("All 4 gold tables written successfully")

2026-04-18 18:26:54,932 [INFO] Writing gold table: nyc_taxi_prod.gold.daily_trip_summary
2026-04-18 18:26:59,580 [INFO] Created new table: nyc_taxi_prod.gold.daily_trip_summary
2026-04-18 18:26:59,581 [INFO] Writing gold table: nyc_taxi_prod.gold.vendor_performance
2026-04-18 18:27:02,592 [INFO] Created new table: nyc_taxi_prod.gold.vendor_performance
2026-04-18 18:27:02,594 [INFO] Writing gold table: nyc_taxi_prod.gold.payment_analysis
2026-04-18 18:27:05,199 [INFO] Created new table: nyc_taxi_prod.gold.payment_analysis
2026-04-18 18:27:05,200 [INFO] Writing gold table: nyc_taxi_prod.gold.hourly_demand
2026-04-18 18:27:07,992 [INFO] Created new table: nyc_taxi_prod.gold.hourly_demand
2026-04-18 18:27:07,993 [INFO] All 4 gold tables written successfully


In [0]:
logger.info("Running VACUUM on all gold tables")

gold_tables = [
    "daily_trip_summary",
    "vendor_performance", 
    "payment_analysis",
    "hourly_demand"
]

for table in gold_tables:
    spark.sql(f"VACUUM nyc_taxi_prod.gold.{table} RETAIN 168 HOURS")
    logger.info(f"VACUUM complete: {table}")

logger.info("All gold tables vacuumed")

##Watermark Output:

In [0]:
max_watermark = df_silver.agg(F.max("updated_at")).collect()[0][0]

output = {
    "max_watermark": str(max_watermark),
    "rows_processed": row_count
}

logger.info(f"Notebook output: {output}")
dbutils.notebook.exit(json.dumps(output))